In [5]:
from function import *

path_ticker = "./data/ticker"
path_stock_data = "./data/stock"

In [6]:
with open("./name_dict.json", "r") as a:
    name_dict = json.load(a)

In [ ]:
data = get_data(path_stock_data,path_ticker,name_dict)

In [ ]:
tickers=get_ticker(path_ticker,name_dict)

In [ ]:
header = get_headers("VCI", False)
date0 = datetime.today().date() - pd.DateOffset(33)
date0 = int(date0.timestamp())
date0
payload = json.dumps({
    "symbol": "ACB",
    "limit": 100,
    "truncTime": None,
})
import requests

_INTRADAY_URL = "market-watch"
base_url = "https://mt.vietcap.com.vn/api/"
url = f"{base_url}{_INTRADAY_URL}/LEData/getAll"
response = requests.post(url, headers=header, data=payload)
data = response.json()
df = pd.DataFrame(data)
df.dtypes

In [12]:
def update_intra_data(path_ticker,path,dictionary):
    tickers=get_ticker(path_ticker,dictionary)
    fal_tick=[]
    data=pd.DataFrame()
    for tick, ex in zip(tickers.symbol, tickers.exchange):
        try:
                temp = read_1_ticker_intra(
                    tick,
                    dictionary
                )
                temp["ticker"] = tick
                temp["exchange"] = ex
                data = pd.concat([data, temp], axis=0)
        except Exception as e:
            fal_tick.append(tick)
    table = pa.Table.from_pandas(data)
    pq.write_to_dataset(table, path)

In [21]:
def read_intra_data(path_ticker,path,dictionary,update=False):
    vietnam_time = datetime.now(pytz.timezone('Asia/Ho_Chi_Minh')).time()
    target_time = datetime.strptime('16:00:00', '%H:%M:%S').time()
    if vietnam_time>target_time and update:
        update_intra_data(path_ticker,path,dictionary)
    data=pd.read_parquet(path)
    return data

In [ ]:
update_intra_data(path_ticker,'./data/intra/',name_dict)

In [ ]:
df=read_intra_data(path_ticker,'./data/intra/',name_dict,update=False)

In [33]:
df[df['ticker']=='VNM']

,time,price,volume,match_type,id,ticker,exchange
5073,2024-10-31 09:15:01,66600.0,26300,ATO/ATC,249548965,VNM,HSX
5072,2024-10-31 09:15:16,66600.0,500,Buy,249549234,VNM,HSX
5071,2024-10-31 09:15:26,66600.0,100,Buy,249549400,VNM,HSX
5070,2024-10-31 09:16:05,66600.0,100,Buy,249550267,VNM,HSX
5069,2024-10-31 09:16:14,66600.0,100,Buy,249550468,VNM,HSX
...,...,...,...,...,...,...,...
4,2024-10-31 13:38:05,65700.0,200,Sell,249724840,VNM,HSX
3,2024-10-31 13:38:05,65700.0,100,Sell,249724841,VNM,HSX
2,2024-10-31 13:38:05,65700.0,300,Sell,249724842,VNM,HSX
1,2024-10-31 13:38:05,65700.0,400,Sell,249724843,VNM,HSX
